In [2]:
import os
import tempfile
from datasets import load_dataset

with tempfile.TemporaryDirectory() as tmp_cache:
    os.environ["HF_DATASETS_CACHE"] = tmp_cache
    dataset = load_dataset("super_glue", "wic", split="train", trust_remote_code=True, streaming=True)
    data = []
    for idx, entry in enumerate(dataset):
        if idx >= 30:
            break
        data.append((entry["sentence1"], entry["sentence2"], entry["label"]))
    print(data)


[('Do you want to come over to my place later?', 'A political system with no place for the less prominent groups.', 0), ('Approach a task.', 'To approach the city.', 0), ('Run rogue.', 'She ran 10 miles that day.', 0), ('The general ordered the colonel to hold his position at all costs.', 'Hold the taxi.', 1), ('We like to summer in the Mediterranean.', 'We summered in Kashmir.', 1), ('His horse won by a head.', 'He is two heads taller than his little sister.', 1), ('The company agrees to meet the cost of any repairs.', 'This proposal meets my requirements.', 1), ('The organism has reached a crucial stage in its development.', 'Our news team brings you the latest developments.', 0), ('The problem with achievement tests is the narrowness they impose on students.', "Frustrated by the narrowness of people's horizons.", 1), ('The governor should act on the new energy bill.', 'Think before you act.', 1), ('Cover her face with a handkerchief.', 'Count the cash in the drawer twice just to cov

In [15]:
import pandas as pd 

data = pd.read_pickle("synthetic_data_controlled.pkl")
data


,examples,word,semantic_group_id
semantic_group_id,,,
0,"[There's a bank in the river., 1. The children...",bank,0
1,"[She went to the bank to open a new account., ...",bank,1
2,"[The bat flew out of the cave., 1. The bat swo...",bat,2
3,"[He used a bat to hit the baseball., 1. The ba...",bat,3
4,"[The dog began to bark loudly., 1. The cat ran...",to,4
5,"[The tree's bark was rough to the touch., 1. T...",to,5
6,"[The pupil in the eye dilated., 1. The doctor ...",pupil,6
7,"[The school pupil studied diligently., 1. The ...",pupil,7
8,"[The metal coil is a spring., The mattress was...",spring,8


In [16]:
data['examples'][8]

['The metal coil is a spring.',
 'The mattress was supported by a set of springs.',
 'She replaced the broken spring in the old clock to restore its chime.',
 'The trampoline was so bouncy it felt like I was jumping on a giant spring.',
 'The clock chimed signaling the arrival of spring.',
 'Engineers tested each coil spring to make sure it could bear heavy loads.',
 "The car's suspension system was equipped with new springs for a smoother ride.",
 'The grass in the park was covered in bright yellow daffodils a sure sign of spring.',
 'The bungee cord stretched like a spring as I prepared to jump off the bridge.',
 'The spring in the pen had lost its tension causing the ink to splatter.',
 'The old jack-in-the-box toy still had the power to make me jump even after all these years.']

In [20]:
data['examples'][0]

["There's a bank in the river.",
 '1. The children loved to play in the shallow bank of the river.\n2. We can have a picnic on the grassy bank of the river.\n3. The bank of the river was crowded with fishermen trying to catch salmon.\n4. The bank of the river was lined with beautiful cherry blossom trees.\n5. The water level in the bank of the river rose after heavy rainfall.\n6. The bank of the river was steep and made it difficult to cross.\n7. The bank of the river was a popular spot for birdwatching.\n8. The old wooden bridge crossed over the narrow bank of the river.\n9. The bank of the river was a perfect place to spot beavers building their dam.\n10. The bank of the river was a peaceful and serene place to relax and unwind.']

In [23]:
import re
import pandas as pd

# 1) Reload
data = pd.read_pickle("synthetic_data_controlled.pkl")

# 2) A bulletproof cleaner
def robust_split(examples):
    # A) Flatten everything into one string
    if isinstance(examples, list):
        blob = " ".join(examples)
    else:
        blob = examples

    # B) Strip out any numbering like "1. ", "2. " etc.
    blob = re.sub(r"\d+\.\s*", "", blob)

    # C) Split on true sentence boundaries (after . ! or ? plus whitespace)
    parts = re.split(r"(?<=[\.\!\?])\s+", blob)

    # D) Trim and drop empty strings
    return [p.strip() for p in parts if p.strip()]

# 3) Apply across the entire column
data['examples'] = data['examples'].apply(robust_split)

# 4) Quick sanity checks
for idx in [0, 1, 8]:
    print(f"Group {idx} →", data.at[idx, 'examples'])
    print("  →", len(data.at[idx, 'examples']), "sentences\n")

# 5) If you still spot a bad sentence, you can manually replace in that list:
#    ex_list = data.at[8, 'examples']
#    ex_list[–1] = "Your corrected sentence with spring."
#    data.at[8, 'examples'] = ex_list

# 6) Save
data.to_pickle("synthetic_data_controlled.pkl")
print("✅ Done: every cell is now a list of clean, split sentences.")

Group 0 → ["There's a bank in the river.", 'The children loved to play in the shallow bank of the river.', 'We can have a picnic on the grassy bank of the river.', 'The bank of the river was crowded with fishermen trying to catch salmon.', 'The bank of the river was lined with beautiful cherry blossom trees.', 'The water level in the bank of the river rose after heavy rainfall.', 'The bank of the river was steep and made it difficult to cross.', 'The bank of the river was a popular spot for birdwatching.', 'The old wooden bridge crossed over the narrow bank of the river.', 'The bank of the river was a perfect place to spot beavers building their dam.', 'The bank of the river was a peaceful and serene place to relax and unwind.']
  → 11 sentences

Group 1 → ['She went to the bank to open a new account.', 'He deposited his paycheck at the bank this morning.', "The bank was closed for the holiday so she couldn't withdraw any money.", 'They needed a loan to start their business so they wen

In [24]:
data['examples'][0]

["There's a bank in the river.",
 'The children loved to play in the shallow bank of the river.',
 'We can have a picnic on the grassy bank of the river.',
 'The bank of the river was crowded with fishermen trying to catch salmon.',
 'The bank of the river was lined with beautiful cherry blossom trees.',
 'The water level in the bank of the river rose after heavy rainfall.',
 'The bank of the river was steep and made it difficult to cross.',
 'The bank of the river was a popular spot for birdwatching.',
 'The old wooden bridge crossed over the narrow bank of the river.',
 'The bank of the river was a perfect place to spot beavers building their dam.',
 'The bank of the river was a peaceful and serene place to relax and unwind.']

In [25]:
data['examples'][1]

['She went to the bank to open a new account.',
 'He deposited his paycheck at the bank this morning.',
 "The bank was closed for the holiday so she couldn't withdraw any money.",
 'They needed a loan to start their business so they went to the bank for help.',
 'The bank teller greeted her with a smile and asked how she could assist.',
 'The bank manager was impressed with her financial plan and approved her loan.',
 'They decided to switch banks due to the high fees at their current one.',
 'She forgot her debit card at home so she had to go to the bank to withdraw cash.',
 "The bank's online banking system was down causing inconvenience for many customers.",
 "The bank's security guard stopped the suspicious-looking man from entering.",
 'After years of saving they finally had enough money to buy their dream home without a mortgage from the bank.']

In [26]:
data['examples'][2]

['The bat flew out of the cave.',
 'The bat swooped down from the tree branch.',
 'I saw a bat hanging upside down in the attic.',
 'The baseball player swung the bat with all his might.',
 'The vampire bat drank blood from its prey.',
 'A bat startled me when it flew past my head.',
 "The bat's echolocation helped it navigate through the dark cave.",
 "The bat's wingspan was impressive as it soared through the sky.",
 'The fruit bat ate pieces of fruit while hanging from the tree.',
 'We walked through the bat exhibit at the zoo.',
 'The cricket player hit a home run with his trusty bat.']

In [27]:
import pandas as pd

# 1) Load
data = pd.read_pickle("synthetic_data_controlled.pkl")

# 2) Choose your bat‐sense‐animal group ID
group_id = 2   # ← replace with your actual index for the animal‐bat group

# 3) Inspect current examples
examples = data.at[group_id, 'examples']
for i, sent in enumerate(examples):
    print(i, sent)

# 4) Suppose you see that examples[3] ("The baseball player…") 
#    and examples[10] ("The cricket player…") are the wrong contexts.
#    Replace them with two new animal‐bat sentences:

examples[3] = "A lone bat fluttered silently against the cavern wall at dusk."
examples[10] = "Biologists tagged each bat before releasing it back into the night sky."

# 5) Put the list back and save
data.at[group_id, 'examples'] = examples
data.to_pickle("synthetic_data_controlled.pkl")

print("✅ Replaced two bad examples in group", group_id)
print("New list:", data.at[group_id, 'examples'])

0 The bat flew out of the cave.
1 The bat swooped down from the tree branch.
2 I saw a bat hanging upside down in the attic.
3 The baseball player swung the bat with all his might.
4 The vampire bat drank blood from its prey.
5 A bat startled me when it flew past my head.
6 The bat's echolocation helped it navigate through the dark cave.
7 The bat's wingspan was impressive as it soared through the sky.
8 The fruit bat ate pieces of fruit while hanging from the tree.
9 We walked through the bat exhibit at the zoo.
10 The cricket player hit a home run with his trusty bat.
✅ Replaced two bad examples in group 2
New list: ['The bat flew out of the cave.', 'The bat swooped down from the tree branch.', 'I saw a bat hanging upside down in the attic.', 'A lone bat fluttered silently against the cavern wall at dusk.', 'The vampire bat drank blood from its prey.', 'A bat startled me when it flew past my head.', "The bat's echolocation helped it navigate through the dark cave.", "The bat's wingsp

In [28]:
data['examples'][3]

['He used a bat to hit the baseball.',
 'The bat flew out of the cave at dusk.',
 'She swung the bat with all her might.',
 'The baseball player held the bat tightly in his hands.',
 'We heard the bat squeaking in the attic.',
 'He hit the ball out of the park with his bat.',
 'The coach handed each player a new bat before the game.',
 'The crowd cheered as the batter connected with the bat.',
 "The bat's wingspan was surprisingly large.",
 'The batter adjusted his grip on the bat before stepping up to the plate.',
 'The bat swooped down and caught the insect mid-air.']

In [29]:
import pandas as pd

# 1) Load your data
data = pd.read_pickle("synthetic_data_controlled.pkl")

# 2) Prepare two correct lists of 11 sentences each:

animal_bat = [
    "The bat flew out of the cave at dusk.",
    "I saw a bat hanging upside down in the attic.",
    "The vampire bat drank blood from its prey.",
    "A bat startled me when it flew past my head.",
    "The bat's echolocation helped it navigate through the dark cave.",
    "The bat's wingspan was impressive as it soared through the sky.",
    "The fruit bat ate pieces of fruit while hanging from the tree.",
    "We walked through the bat exhibit at the zoo.",
    "Biologists tagged each bat before releasing it back into the night sky.",
    "They photographed a colony of bats emerging at twilight from a gorge.",
    "The tiny bat clung to the cave wall by its clawed feet."
]

sports_bat = [
    "He used a bat to hit the baseball.",
    "She swung the bat with all her might.",
    "The baseball player held the bat tightly in his hands.",
    "He hit the ball out of the park with his bat.",
    "The coach handed each player a new bat before the game.",
    "The crowd cheered as the batter connected with the bat.",
    "The batter adjusted his grip on the bat before stepping up to the plate.",
    "She admired the engraved name on her new oak bat.",
    "Every player sanded their bat's surface for a better grip.",
    "He tested the weight of the bat before the first pitch.",
    "The custom bat's handle was wrapped in black tape."
]

# 3) Overwrite the two “bat” groups in your DataFrame.
#    (Replace 2 and 3 below with your actual group IDs.)

data.at[2, 'examples'] = animal_bat
data.at[3, 'examples'] = sports_bat

# 4) Quick sanity check
print("Animal‐bat group:", data.at[2, 'examples'])
print("Sports‐bat group:", data.at[3, 'examples'])

# 5) Save back
data.to_pickle("synthetic_data_controlled.pkl")
print("✅ ‘bat’ contexts have been corrected.") 

Animal‐bat group: ['The bat flew out of the cave at dusk.', 'I saw a bat hanging upside down in the attic.', 'The vampire bat drank blood from its prey.', 'A bat startled me when it flew past my head.', "The bat's echolocation helped it navigate through the dark cave.", "The bat's wingspan was impressive as it soared through the sky.", 'The fruit bat ate pieces of fruit while hanging from the tree.', 'We walked through the bat exhibit at the zoo.', 'Biologists tagged each bat before releasing it back into the night sky.', 'They photographed a colony of bats emerging at twilight from a gorge.', 'The tiny bat clung to the cave wall by its clawed feet.']
Sports‐bat group: ['He used a bat to hit the baseball.', 'She swung the bat with all her might.', 'The baseball player held the bat tightly in his hands.', 'He hit the ball out of the park with his bat.', 'The coach handed each player a new bat before the game.', 'The crowd cheered as the batter connected with the bat.', 'The batter adjus

In [39]:
data

,examples,word,semantic_group_id
semantic_group_id,,,
0,"[There's a bank in the river., The children lo...",bank,0
1,"[She went to the bank to open a new account., ...",bank,1
2,"[The bat flew out of the cave at dusk., I saw ...",bat,2
3,"[He used a bat to hit the baseball., She swung...",bat,3
4,[This laptop is so light I can carry it with o...,light,4
5,[Morning light streamed through the bedroom wi...,light,5
6,"[The pupil in the eye dilated., The doctor sho...",pupil,6
7,"[The school pupil studied diligently., The you...",pupil,7
8,"[The metal coil is a spring., The mattress was...",spring,8


In [34]:
import pandas as pd

# 1) Load your DataFrame
data = pd.read_pickle("synthetic_data_controlled.pkl")

# 2) Define your two new example‐lists for “light”
light_not_heavy = [
    "This laptop is so light I can carry it with one hand.",
    "She packed a light backpack for the day hike.",
    "The new running shoes feel incredibly light on my feet.",
    "I prefer a light jacket in the spring breeze.",
    "His feather-light touch made the sculpture seem to float.",
    "They installed a light folding table for easy storage.",
    "The drone’s light carbon-fiber frame improved its flight time.",
    "She chose a light ceramic vase to avoid stressing the shelf.",
    "The artist used a light canvas to hang her painting.",
    "He bought a light aluminum canoe for weekend trips."
]

light_illumination = [
    "Morning light streamed through the bedroom window.",
    "She adjusted the lamp to cast a softer light on her book.",
    "The streetlight flickered in the misty night air.",
    "They followed the faint light at the end of the tunnel.",
    "Candlelight danced across the walls during the storm.",
    "He waited for the light to change before crossing the road.",
    "The light of the full moon illuminated the quiet lake.",
    "A beam of light broke through the darkened forest canopy.",
    "The projector’s bright light filled the cinema screen.",
    "She admired the northern lights shimmering in the sky."
]

# 3) Find the two semantic_group_ids where word == 'bark'
bark_groups = data.index[data['word'] == 'to'].tolist()
print("Bark groups found at IDs:", bark_groups)

# 4) Overwrite them with your new “light” groups
#    We’ll also update the 'word' column to 'light'
data.at[bark_groups[0], 'word']     = 'light'
data.at[bark_groups[0], 'examples'] = light_not_heavy

data.at[bark_groups[1], 'word']     = 'light'
data.at[bark_groups[1], 'examples'] = light_illumination

# 5) Sanity check
print("\nReplaced Group", bark_groups[0], "→", data.at[bark_groups[0], 'examples'])
print("\nReplaced Group", bark_groups[1], "→", data.at[bark_groups[1], 'examples'])

# 6) Save back
data.to_pickle("synthetic_data_controlled.pkl")
print("\n✅ ‘bark’ has been replaced with two ‘light’ sense groups.")

Bark groups found at IDs: [4, 5]

Replaced Group 4 → ['This laptop is so light I can carry it with one hand.', 'She packed a light backpack for the day hike.', 'The new running shoes feel incredibly light on my feet.', 'I prefer a light jacket in the spring breeze.', 'His feather-light touch made the sculpture seem to float.', 'They installed a light folding table for easy storage.', 'The drone’s light carbon-fiber frame improved its flight time.', 'She chose a light ceramic vase to avoid stressing the shelf.', 'The artist used a light canvas to hang her painting.', 'He bought a light aluminum canoe for weekend trips.']

Replaced Group 5 → ['Morning light streamed through the bedroom window.', 'She adjusted the lamp to cast a softer light on her book.', 'The streetlight flickered in the misty night air.', 'They followed the faint light at the end of the tunnel.', 'Candlelight danced across the walls during the storm.', 'He waited for the light to change before crossing the road.', 'The

In [37]:
data['examples'][6]

['The pupil in the eye dilated.',
 'The doctor shone a light in her pupil to check for any abnormalities.',
 'The pupil of his eye contracted in response to the bright sunlight.',
 'The teacher noticed that one pupil was struggling to read the small print.',
 'The ophthalmologist examined the pupil for signs of glaucoma.',
 'The dilated pupil indicated that the patient had received a sedative.',
 'The optometrist measured the size of the pupil to determine the correct prescription.',
 "The pupil's eye widened in surprise as the magician pulled a rabbit out of his hat.",
 'The doctor used a special tool to measure the pressure in the pupil of the eye.',
 "The student's dilated pupils suggested that he was under the influence of drugs.",
 'The photographer asked the model to look directly into the camera to capture her pupils in the shot.']

In [38]:
data['examples'][7]

['The school pupil studied diligently.',
 "The young pupil eagerly raised their hand to answer the teacher's question.",
 'The diligent pupil took detailed notes during the science experiment.',
 "The school's top pupil received a scholarship for their outstanding academic performance.",
 "The pupil's parents were proud of their child's achievements in school.",
 'The teacher praised the pupil for their hard work and dedication.',
 "The pupil's grades showed a significant improvement after receiving extra help from a tutor.",
 'The curious pupil asked insightful questions during the history lesson.',
 'The shy pupil slowly gained confidence and actively participated in class discussions.',
 "The school's best pupil was chosen to represent their class in a spelling bee competition.",
 "The pupil's determination and perseverance paid off when they received the top award at graduation."]

In [ ]:
import pandas as pd

# 1) Load your DataFrame
data = pd.read_pickle("synthetic_data_controlled.pkl")

# 2) Define your two new example‐lists for “light”
light_not_heavy = [
    "This laptop is so light I can carry it with one hand.",
    "She packed a light backpack for the day hike.",
    "The new running shoes feel incredibly light on my feet.",
    "I prefer a light jacket in the spring breeze.",
    "His feather-light touch made the sculpture seem to float.",
    "They installed a light folding table for easy storage.",
    "The drone’s light carbon-fiber frame improved its flight time.",
    "She chose a light ceramic vase to avoid stressing the shelf.",
    "The artist used a light canvas to hang her painting.",
    "He bought a light aluminum canoe for weekend trips."
]

light_illumination = [
    "Morning light streamed through the bedroom window.",
    "She adjusted the lamp to cast a softer light on her book.",
    "The streetlight flickered in the misty night air.",
    "They followed the faint light at the end of the tunnel.",
    "Candlelight danced across the walls during the storm.",
    "He waited for the light to change before crossing the road.",
    "The light of the full moon illuminated the quiet lake.",
    "A beam of light broke through the darkened forest canopy.",
    "The projector’s bright light filled the cinema screen.",
    "She admired the northern lights shimmering in the sky."
]

# 3) Find the two semantic_group_ids where word == 'bark'
bark_groups = data.index[data['word'] == 'to'].tolist()
print("Bark groups found at IDs:", bark_groups)

# 4) Overwrite them with your new “light” groups
#    We’ll also update the 'word' column to 'light'
data.at[bark_groups[0], 'word']     = 'light'
data.at[bark_groups[0], 'examples'] = light_not_heavy

data.at[bark_groups[1], 'word']     = 'light'
data.at[bark_groups[1], 'examples'] = light_illumination

# 5) Sanity check
print("\nReplaced Group", bark_groups[0], "→", data.at[bark_groups[0], 'examples'])
print("\nReplaced Group", bark_groups[1], "→", data.at[bark_groups[1], 'examples'])

# 6) Save back
data.to_pickle("synthetic_data_controlled.pkl")
print("\n✅ ‘bark’ has been replaced with two ‘light’ sense groups.")

In [40]:
# 1) Load your DataFrame
data = pd.read_pickle("synthetic_data_controlled.pkl")
data

,examples,word,semantic_group_id
semantic_group_id,,,
0,"[There's a bank in the river., The children lo...",bank,0
1,"[She went to the bank to open a new account., ...",bank,1
2,"[The bat flew out of the cave at dusk., I saw ...",bat,2
3,"[He used a bat to hit the baseball., She swung...",bat,3
4,[This laptop is so light I can carry it with o...,light,4
5,[Morning light streamed through the bedroom wi...,light,5
6,"[The pupil in the eye dilated., The doctor sho...",pupil,6
7,"[The school pupil studied diligently., The you...",pupil,7
8,"[The metal coil is a spring., The mattress was...",spring,8


In [44]:
import pandas as pd

# 1) Load your DataFrame
data = pd.read_pickle("synthetic_data_controlled.pkl")

# 2) Define your two new example‐lists for “too”
too_also = [
    "I too have a suggestion for improving the design.",
    "She, too, decided to postpone her trip until next week.",
    "They too received an invitation to the exclusive gala.",
    "He too can join the team once he completes his training.",
    "Our neighbors too started renovating their house this month.",
    "They brought pizza for the meeting too.",
    "You should read this book too.",
    "They enjoyed the movie, and I did too.",
    "You can have some cake too.",
    "If you’re going, I’d like to come too."
]

too_excessive = [
    "This soup is too hot to eat.",
    "The movie was too long for my taste.",
    "You’re driving too fast on this road.",
    "I’ve had too much sugar today.",
    "The water is too cold for swimming.",
    "She’s talking too loudly in the library.",
    "He spends too much time on his phone.",
    "The bag is too heavy for me to lift.",
    "This problem is too difficult to solve quickly.",
    "The coffee is too bitter without cream."
]

# 3) Find the two semantic_group_ids where word == 'too'
too_groups = data.index[data['word'] == 'pupil'].tolist()
print("‘too’ groups found at IDs:", too_groups)

# 4) Overwrite them with your new “too” groups
#    (we keep word == 'too', just swap the examples)
data.at[too_groups[0], 'word']     = 'too'
data.at[too_groups[0], 'examples'] = too_also
data.at[too_groups[1], 'word']     = 'too'
data.at[too_groups[1], 'examples'] = too_excessive

# 5) Sanity check
print(f"\nReplaced Group {too_groups[0]} (also) →", data.at[too_groups[0], 'examples'])
print(f"\nReplaced Group {too_groups[1]} (excessively) →", data.at[too_groups[1], 'examples'])

# 6) Save back
data.to_pickle("synthetic_data_controlled.pkl")
print("\n✅ ‘too’ has been refreshed with correct ‘also’ vs ‘excessively’ examples.")

‘too’ groups found at IDs: [6, 7]

Replaced Group 6 (also) → ['I too have a suggestion for improving the design.', 'She, too, decided to postpone her trip until next week.', 'They too received an invitation to the exclusive gala.', 'He too can join the team once he completes his training.', 'Our neighbors too started renovating their house this month.', 'They brought pizza for the meeting too.', 'You should read this book too.', 'They enjoyed the movie, and I did too.', 'You can have some cake too.', 'If you’re going, I’d like to come too.']

Replaced Group 7 (excessively) → ['This soup is too hot to eat.', 'The movie was too long for my taste.', 'You’re driving too fast on this road.', 'I’ve had too much sugar today.', 'The water is too cold for swimming.', 'She’s talking too loudly in the library.', 'He spends too much time on his phone.', 'The bag is too heavy for me to lift.', 'This problem is too difficult to solve quickly.', 'The coffee is too bitter without cream.']

✅ ‘too’ has

In [45]:
data = pd.read_pickle("synthetic_data_controlled.pkl")
data

,examples,word,semantic_group_id
semantic_group_id,,,
0,"[There's a bank in the river., The children lo...",bank,0
1,"[She went to the bank to open a new account., ...",bank,1
2,"[The bat flew out of the cave at dusk., I saw ...",bat,2
3,"[He used a bat to hit the baseball., She swung...",bat,3
4,[This laptop is so light I can carry it with o...,light,4
5,[Morning light streamed through the bedroom wi...,light,5
6,[I too have a suggestion for improving the des...,too,6
7,"[This soup is too hot to eat., The movie was t...",too,7
8,"[The metal coil is a spring., The mattress was...",spring,8
